# CIFAKE-Guard v2: Google Colab runner

This notebook runs the **full-data protocol** (`configs/full_gpu.yaml`) of CIFAKE-Guard v2 on a Colab GPU.
All logic lives in the `cifake_guard` package and the `scripts/`; the notebook only orchestrates.

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

Steps: 1) get the code, 2) install, 3) audit the data, 4) train, 5) build the unseen-generator benchmark,
6) evaluate everything with one protocol, 7) inspect results, 8) (optional) launch the dashboard, 9) download outputs.

## 1. Get the project code
Option A: upload `CIFAKE_Guard_v2.zip` (the project folder zipped) with the file browser on the left, then run the cell.
Option B: put the folder on Google Drive and set `PROJECT_DIR`.

In [ ]:
import os, zipfile, pathlib
PROJECT_DIR = "/content/CIFAKE_Guard"          # change if you use Google Drive
if not pathlib.Path(PROJECT_DIR, "cifake_guard").exists() and pathlib.Path("/content/CIFAKE_Guard_v2.zip").exists():
    zipfile.ZipFile("/content/CIFAKE_Guard_v2.zip").extractall(PROJECT_DIR)
    # the zip may contain a top-level folder
    inner = [p for p in pathlib.Path(PROJECT_DIR).iterdir() if (p / "cifake_guard").exists()]
    if inner: PROJECT_DIR = str(inner[0])
os.chdir(PROJECT_DIR); print("Working in", os.getcwd()); print(sorted(os.listdir()))

## 2. Install dependencies
Colab already ships PyTorch; this installs the rest.

In [ ]:
!pip install -q -r requirements.txt
import torch; print(torch.__version__, "CUDA:", torch.cuda.is_available())

## 3. Dataset audit (format shortcuts, exact & near duplicates)
Downloads CIFAKE through `kagglehub` (~105 MB, no credentials) and decodes it once into `data_cache/`.

In [ ]:
!python scripts/audit_data.py --config configs/full_gpu.yaml --out results

In [ ]:
import json; a = json.load(open("results/data_audit.json"))
print(json.dumps(a["exact_duplicates"]["by_location"], indent=1))
print({k: v for k, v in a["near_duplicates_test_vs_train"].items() if k != "examples"})

## 4. Train all models
Loss/accuracy curves are refreshed in `runs/full_gpu/plots/` after every epoch. Approximate T4 time: 1-3 h for everything. Train a subset with `--models LightCNN SFHybrid`.

In [ ]:
!python scripts/train.py --config configs/full_gpu.yaml

## 5. Unseen-generator benchmark (DDPM trained on CIFAR-10)
Fast on GPU; produces 2000 fake + 2000 real images stored exactly like CIFAKE (32x32 JPEG q75).

In [ ]:
!pip install -q diffusers
!python scripts/generate_unseen_fakes.py --n 2000 --steps 50 --batch 250 --out data_unseen/ddpm_cifar10

## 6. Unified evaluation
Calibration, ensembles, bootstrap CIs, de-duplicated test, robustness, unseen generator, latency, explanation faithfulness.

In [ ]:
!python scripts/evaluate.py --config configs/full_gpu.yaml --unseen data_unseen/ddpm_cifar10

## 7. Results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("runs/full_gpu/results/RESULTS.md").read()))
for p in ["roc_pr_curves", "calibration_curves", "robustness_accuracy", "residual_spectra", "confusion_matrices", "accuracy_vs_cost"]:
    display(Image(f"runs/full_gpu/plots/{p}.png"))

## 8. (Optional) Launch the dashboard inside Colab

In [ ]:
import subprocess, time, os
env = dict(os.environ, CIFAKE_RUN_DIR="runs/full_gpu")
server = subprocess.Popen(["python", "-m", "uvicorn", "app.main:app", "--port", "8000"], env=env)
time.sleep(15)
from google.colab.output import eval_js
print("Open:", eval_js("google.colab.kernel.proxyPort(8000)"))

## 9. Download outputs
Colab's disk is wiped when the runtime ends.

In [ ]:
import shutil
shutil.make_archive("cifake_guard_full_gpu_run", "zip", "runs/full_gpu")
from google.colab import files; files.download("cifake_guard_full_gpu_run.zip")